# 3. Simulate supernovae I and II and Kilonovae

- Test https://skysurvey.readthedocs.io/en/latest/transientclasses/sne_ia.html

- **Author:** Sylvie Dagoret-Campagne
- **Affiliation:** IJCLab/IN2P3/CNRS
- **Creation date:** 2026-08-13
- **Last update** 2026-08-13 
- **mac**: python kernel = conda_py313

In [ ]:
import skysurvey
from skysurvey.target.snia import SNeIaColor

import os

## LSST Survey

In [ ]:
# RUBIN_SIM_DATA_DIR points to the local cache of rubin_sim/rubin_scheduler auxiliary data
# (opsim databases, dust maps, SN gamma/noise files, throughputs, ...).
# os.environ["RUBIN_SIM_DATA_DIR"] = "/users/dagoret/DATA/OpSim"
PATH_OPSIM = os.getenv("RUBIN_SIM_DATA_DIR")
print(f"PATH_OPSIM = {PATH_OPSIM}")

In [ ]:
file_opsim = "sim_baseline/baseline_v5.3.5_10yrs.db"
# file_opsim = "ddf_one_less_v5.3.2_10yrs.db"
# file_opsim = "ddf_sd_v5.3.0_10yrs.db"

In [ ]:
# lsst opsim files are large, this may take a few minutes
opsim_path = os.path.join(PATH_OPSIM, file_opsim)
survey_lsst = skysurvey.LSST.from_opsim(opsim_path)

In [ ]:
survey_lsst.data["band"] = survey_lsst.data["band"].str.split("_").str[0]

In [ ]:
survey_lsst.data.head()

## SNIa

In [ ]:
snia = skysurvey.SNeIa()

In [ ]:
snia

#### Template

In [ ]:
snia.template

#### Rate

Each pre-computed skysurvey transient has a default volumetric rate, that provide the number of transient expected per year and per Gpc³. The SNeIa rate is derived from Perley et al.(2020) (2020ApJ…904…35P)

In [ ]:
snia.rate

#### Model

In [ ]:
snia.template.source

The model is based on the modeldag package.

For the SNeIa class, the model contains 8 entries. So the generated data will contains at least 8 columns. To display the model, you can directly print the object:

In [ ]:
snia.template.parameters

In [ ]:
snia.model

#### View

In [ ]:
import matplotlib.pyplot as plt


xx, pdf = SNeIaColor.intrinsic_and_dust()
fig = plt.figure(figsize=[6, 4])
plt.plot(xx, pdf, color="darkred")
plt.xlabel("c")
plt.ylabel("PDF")
plt.show()

#### Data

In [ ]:
data_snia = snia.draw(size=10_000)

In [ ]:
data_snia

## SNII

In skysurvey, Core-Collapse (CC) SNe, including Type II, IIn, IIb, Ib, Ic, and Ic-BL, are pre-built classes that inherit from a common base.


Their class structure are defined by:

    VincenziModels, which provides the link to Vincenzi et al. (2019) (2019MNRAS.489.5802V) time-series models implemented in sncosmo.

    MultiTemplateTSTransient, which allows combining multiple templates to model the diversity of observed CC SNe.

    Each subtype inherits its own parameters: _KIND, _RATE, _MAGABS, and a set of v19-*-corr templates.



In [ ]:
snii = skysurvey.SNeII()

#### Templates

The different CC SNe classes are associated with time-series templates from Vincenzi et al. (2019), implemented in sncosmo under the v19-*-corr name, and stored in skysurvey as a TemplateCollection object.

In [ ]:
snii.template

In [ ]:
snii.template.names

#### Rate

In [ ]:
snii.rate

#### Models

In [ ]:
snii.model

#### Data

In [ ]:
data_snii = snii.draw(size=10_000)

In [ ]:
data_snii

## Kilonovae

In skysurvey, kilonovae are a pre-built class modeled from a spectra model computed using the POSSIS code from Bulla 2019 (2019MNRAS.489.5037B) and converted as spectral time series into sncosmo, making kilonovae usable like any other transient type.


In [ ]:
kilonova = skysurvey.Kilonova()

#### Template

In [ ]:
kilonova.template

#### Rate

In [ ]:
kilonova.rate

#### Models

In [ ]:
kilonova.model

## 4. Demo: example light curves for SNe Ia, SLSN, SNe II, and Kilonovae

**Why `show_lightcurve` was failing above.** `show_lightcurve(band, index=...)` always reads the
target's drawn parameters from `self.data` (via `get_target_template` -> `get_template`, which
defaults to `data=self.data` internally) -- **not** from whatever `DataFrame` `.draw()` happened to
return. `Target.draw(...)` only sets `self.data` if you pass `inplace=True`; otherwise it just
*returns* the drawn `DataFrame` and leaves `self.data` as `None`. That is exactly what happens in
the `## SNIa` section above: `data_snia = snia.draw(size=10_000)` returns a perfectly good
`DataFrame`, but `snia.data` is still `None`, so `snia.show_lightcurve(...)` fails with
`AttributeError: 'NoneType' object has no attribute 'columns'`. The `TSTransient`/`SNeII` calls
above work because `skysurvey.TSTransient.from_draw(...)` (the *classmethod*, not `.draw()`) does
set `self.data` for you.

**The fix, one line:** either
1. `target.draw(size=N, ..., inplace=True)` -- sets `target.data` in place (used below), or
2. keep `data = target.draw(size=N)` as-is and pass it explicitly:
   `target.show_lightcurve(band, index=idx, params={"data": data})`.

Below: one drawn population per class, then a single example light curve per class, then a 2x2
summary figure. For SLSN we reuse the `nugent-hyper` bright-proxy approach from notebook 11 --
**`sncosmo`/`skysurvey` have no genuine, physically-modeled SLSN template**; see notebook 11,
Section 1bis, for the full caveat.

**Note:** the first time each class is drawn, `sncosmo`/`skysurvey` downloads and caches the
corresponding template data (SALT2/3 for SNe Ia, Vincenzi et al. 2019 `v19-*-corr` for SNe II,
the POSSIS/Bulla grid for kilonovae, Nugent templates for the SLSN proxy) -- this needs network
access the first time, then is cached locally (typically under `~/.astropy` /
`sncosmo`'s data directory).

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import skysurvey

np.random.seed(0)
BANDS = ["lsstg", "lsstr", "lssti"]  # bands to overplot on each light curve

### SNe Ia

In [ ]:
snia_demo = skysurvey.SNeIa()
# inplace=True is the key fix: it sets snia_demo.data, which show_lightcurve() needs.
snia_demo.draw(size=2_000, zmax=0.6, inplace=True)

idx_snia = snia_demo.data["magabs"].idxmin()  # pick the intrinsically brightest one, for a clean example
z_snia = snia_demo.data.loc[idx_snia, "z"]

fig = snia_demo.show_lightcurve(BANDS, index=idx_snia, in_mag=True)
fig.suptitle(f"SNe Ia -- example light curve (index={idx_snia}, z={z_snia:.2f})")

### SLSN (proxy template, see caveat)

No genuine SLSN model exists in `sncosmo`'s registry, so this uses `TSTransient` with the
`nugent-hyper` engine-driven template and a bright `magabs` prior, as an illustrative stand-in
(see notebook 11, Section 1bis). `TSTransient.from_draw(...)` already sets `self.data`, so no
`inplace=True` is needed here.

In [ ]:
slsn_demo = skysurvey.TSTransient.from_draw(
    size=2_000,
    template="nugent-hyper",  # proxy template -- no genuine SLSN source in sncosmo, see caveat above
    magabs=[-21.0, 1.0],  # SLSN-I peak ~ -21, vs ~-18 for a normal Ib/c
    zmax=0.3,
)

idx_slsn = slsn_demo.data.index[0]
z_slsn = slsn_demo.data.loc[idx_slsn, "z"]

fig = slsn_demo.show_lightcurve(BANDS, index=idx_slsn, in_mag=True)
fig.suptitle(f"SLSN proxy (nugent-hyper) -- example light curve (index={idx_slsn}, z={z_slsn:.2f})")

### SNe II

In [ ]:
snii_demo = skysurvey.SNeII()
snii_demo.draw(size=2_000, zmax=0.3, inplace=True)

idx_snii = snii_demo.data.index[0]
z_snii = snii_demo.data.loc[idx_snii, "z"]

fig = snii_demo.show_lightcurve(BANDS, index=idx_snii, in_mag=True)
fig.suptitle(f"SNe II -- example light curve (index={idx_snii}, z={z_snii:.2f})")

### Kilonovae

In [ ]:
kn_demo = skysurvey.Kilonova()
kn_demo.draw(size=2_000, zmax=0.15, inplace=True)

idx_kn = kn_demo.data.index[0]
z_kn = kn_demo.data.loc[idx_kn, "z"]

fig = kn_demo.show_lightcurve(BANDS, index=idx_kn, in_mag=True)
fig.suptitle(f"Kilonova -- example light curve (index={idx_kn}, z={z_kn:.2f})")

### Summary: all four classes side by side

`show_lightcurve` accepts `ax=` (and `fig=`), so the four examples above can be dropped into a
single 2x2 grid for an at-a-glance comparison.

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(11, 8), sharex=False)

snia_demo.show_lightcurve(BANDS, index=idx_snia, in_mag=True, ax=axes[0, 0])
axes[0, 0].set_title(f"SNe Ia (z={z_snia:.2f})")

slsn_demo.show_lightcurve(BANDS, index=idx_slsn, in_mag=True, ax=axes[0, 1])
axes[0, 1].set_title(f"SLSN proxy, nugent-hyper (z={z_slsn:.2f})")

snii_demo.show_lightcurve(BANDS, index=idx_snii, in_mag=True, ax=axes[1, 0])
axes[1, 0].set_title(f"SNe II (z={z_snii:.2f})")

kn_demo.show_lightcurve(BANDS, index=idx_kn, in_mag=True, ax=axes[1, 1])
axes[1, 1].set_title(f"Kilonova (z={z_kn:.2f})")

fig.suptitle("Example light curves: SNe Ia / SLSN (proxy) / SNe II / Kilonova", fontsize=14)
fig.tight_layout()